In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import datetime
import json
import os
import time
import random

In [2]:
# 启动 Chrome 浏览器（带反检测配置）
try:
    driver.quit()  # 如果之前有打开，先关闭
except:
    pass

chrome_options = Options()
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=chrome_options)

# 防自动化检测
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': '''
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        })
    '''
})

print("浏览器已启动")

✅ 浏览器已启动


In [3]:
# 加载并应用 Cookie 登录 Instagram

COOKIE_FILE = "instagram_cookies.json"

# 1. 检查文件是否存在
if not os.path.exists(COOKIE_FILE):
    print(f"错误: 找不到 Cookie 文件 '{COOKIE_FILE}'")
else:
    # 2. 读取 Cookie
    try:
        with open(COOKIE_FILE, 'r', encoding='utf-8') as f:
            cookies = json.load(f)
        print(f"✓ 成功加载 {len(cookies)} 个 Cookie")
    except Exception as e:
        print(f"读取 Cookie 失败: {e}")
        cookies = None

    # 3. 应用 Cookie
    if cookies:
        print("正在应用 Cookie...")
        driver.get("https://www.instagram.com/")
        time.sleep(2)
        driver.delete_all_cookies()
        for cookie in cookies:
            try:
                # 确保 domain 匹配（有些导出工具会带 .instagram.com）
                if 'domain' in cookie and 'instagram.com' in cookie['domain']:
                    driver.add_cookie(cookie)
            except Exception as e:
                continue  # 忽略无效 cookie
        
        print("刷新页面以应用登录状态...")
        driver.refresh()
        time.sleep(random.uniform(3,5))

        # 4. 验证是否登录成功
        try:
            current_url = driver.current_url
            if any(keyword in current_url for keyword in ['/login', '/signin', '/auth']):
                print("当前是登录页面，未登录")
        except:
            print("未检测到登录状态，请检查 Cookie 是否有效。")

✓ 成功加载 12 个 Cookie
正在应用 Cookie...
🔄 刷新页面以应用登录状态...


In [7]:
# 搜索标签并滚动获取帖子链接

# ====== 配置参数（可修改）======
keyword = "powerpuff"       # 不带 #，程序会自动处理
max_posts = 100           # 最多获取多少条
max_scrolls = 10         # 最多滚动多少次
scroll_delay = 3        # 每次滚动后等待几秒
# ==============================

hashtag = keyword.lstrip('#')
search_url = f"https://www.instagram.com/explore/tags/{hashtag}/"
print(f"正在访问标签页: #{hashtag}")
driver.get(search_url)
time.sleep(random.uniform(3,8))

# 初始化全局列表（可在后续 cell 中访问）
post_urls = []

scroll_count = 0
print("开始滚动加载帖子...")

while len(post_urls) < max_posts and scroll_count < max_scrolls:
    scroll_count += 1
    print(f"\n第 {scroll_count} 次滚动 | 当前链接数: {len(post_urls)}")

    # 提取当前页面所有 /p/ 链接
    try:
        links = driver.find_elements(By.XPATH, "//a[contains(@href, '/p/')]")
        for link in links:
            href = link.get_attribute("href")
            if not href:
                continue
            # 清理 URL
            clean_url = href.split('?')[0] if '?' in href else href
            if clean_url.startswith('/'):
                clean_url = "https://www.instagram.com" + clean_url
            if clean_url.startswith('https://www.instagram.com/p/') and clean_url not in post_urls:
                post_urls.append(clean_url)
                if len(post_urls) >= max_posts:
                    break
    except Exception as e:
        print(f"提取链接时出错: {e}")

    # 检查是否达到目标
    if len(post_urls) >= max_posts:
        print("已达到目标数量")
        break

    # 继续滚动
    if scroll_count < max_scrolls:
        print("  下滑页面...")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(scroll_delay)
    else:
        print(" 达到最大滚动次数")

# 截断到目标数量
post_urls = post_urls[:max_posts]
print(f"\n共获取 {len(post_urls)} 条唯一帖子链接")

🔍 正在访问标签页: #powerpuff
🔽 开始滚动加载帖子...

🔄 第 1 次滚动 | 当前链接数: 0
  下滑页面...

🔄 第 2 次滚动 | 当前链接数: 18
  下滑页面...

🔄 第 3 次滚动 | 当前链接数: 18
  下滑页面...

🔄 第 4 次滚动 | 当前链接数: 28
  下滑页面...

🔄 第 5 次滚动 | 当前链接数: 33
  下滑页面...

🔄 第 6 次滚动 | 当前链接数: 35
  下滑页面...

🔄 第 7 次滚动 | 当前链接数: 35
  下滑页面...

🔄 第 8 次滚动 | 当前链接数: 35
  下滑页面...

🔄 第 9 次滚动 | 当前链接数: 35
  下滑页面...

🔄 第 10 次滚动 | 当前链接数: 35
  达到最大滚动次数

🎉 共获取 35 条唯一帖子链接


In [8]:
# 查看获取到的链接（前10条）
if 'post_urls' in globals() and post_urls:
    print(f"共 {len(post_urls)} 条链接，显示前10条：\n")
    for i, url in enumerate(post_urls[:10], 1):
        print(f"{i:2d}. {url}")
    if len(post_urls) > 10:
        print(f"\n... 还有 {len(post_urls) - 10} 条")
else:
    print("尚未获取任何链接，请先运行上一步")

📋 共 35 条链接，显示前10条：

 1. https://www.instagram.com/p/DAQzdDQP5gY/
 2. https://www.instagram.com/p/B9-ASMrl0f8/
 3. https://www.instagram.com/p/DCH354MMmTn/
 4. https://www.instagram.com/p/DNEZSjzsZMY/
 5. https://www.instagram.com/p/DGbHoGjxj3p/
 6. https://www.instagram.com/p/DTg12zgktSa/
 7. https://www.instagram.com/p/DTbFV7wE0OM/
 8. https://www.instagram.com/p/DNiEUM8M6xi/
 9. https://www.instagram.com/p/DMxt-mRqxFm/
10. https://www.instagram.com/p/DNwDs_42COl/

... 还有 25 条


In [9]:
import re
import json
import time
from typing import Optional
from selenium.webdriver.common.by import By
import pandas as pd

# 系统路径黑名单：这些不是用户名
RESERVED_PATHS = {
    "reels", "reel", "explore", "accounts", "direct", "stories", "tv",
    "p", "about", "developer", "privacy", "terms", "api", "web", "emails",
}

def is_valid_username(username: str) -> bool:
    """验证是否为有效的Instagram用户名"""
    if not username:
        return False
    u = username.strip().lower()
    if u in RESERVED_PATHS:
        return False
    # IG 用户名合法字符：字母、数字、点、下划线
    if not re.match(r"^[a-z0-9._]+$", u):
        return False
    return True

def is_valid_profile_url(url: str) -> bool:
    """验证是否为有效的Instagram个人主页URL"""
    if not url:
        return False
    # 匹配 https://www.instagram.com/用户名/ 格式
    m = re.match(r"^https?://(www\.)?instagram\.com/([^/]+)/$", url)
    if not m:
        return False
    username = m.group(2).strip().lower()
    return is_valid_username(username)

def profile_url_from_username(username: str) -> str:
    """根据用户名生成个人主页URL"""
    return f"https://www.instagram.com/{username.strip().lower()}/"

def extract_author_from_jsonld(driver) -> Optional[str]:
    """
    从JSON-LD数据中提取作者用户名
    这是第二解析路径
    """
    try:
        scripts = driver.find_elements(By.CSS_SELECTOR, 'script[type="application/ld+json"]')
        for s in scripts:
            raw = (s.get_attribute("innerText") or "").strip()
            if not raw:
                continue
                
            try:
                data = json.loads(raw)
            except Exception:
                continue

            # data 可能是 dict 或 list
            candidates = []
            if isinstance(data, dict):
                candidates.append(data)
            elif isinstance(data, list):
                candidates.extend([x for x in data if isinstance(x, dict)])

            for obj in candidates:
                # 尝试从 author 字段提取
                author = obj.get("author")
                if isinstance(author, dict):
                    name = author.get("alternateName") or author.get("name")
                    if isinstance(name, str) and is_valid_username(name):
                        return name
                elif isinstance(author, list):
                    for a in author:
                        if isinstance(a, dict):
                            name = a.get("alternateName") or a.get("name")
                            if isinstance(name, str) and is_valid_username(name):
                                return name

                # 尝试从 creator / accountablePerson 字段提取
                creator = obj.get("creator") or obj.get("accountablePerson")
                if isinstance(creator, dict):
                    name = creator.get("alternateName") or creator.get("name")
                    if isinstance(name, str) and is_valid_username(name):
                        return name

    except Exception:
        pass
    return None

def extract_author_from_pagesource_regex(driver) -> Optional[str]:
    """
    从页面源代码中通过正则匹配提取作者用户名
    这是第三兜底方法
    """
    html = driver.page_source or ""
    if not html:
        return None

    patterns = [
        # 常见 owner / user 结构（修复了转义问题）
        r'"owner"\s*:\s*\{[^}]*?"username"\s*:\s*"([^"]+)"',
        r'"user"\s*:\s*\{[^}]*?"username"\s*:\s*"([^"]+)"',
        # 有些结构是 "username":"xxx","full_name":...
        r'"username"\s*:\s*"([^"]+)"\s*,\s*"full_name"',
        # 兜底：但容易误抓，放最后
        r'"username"\s*:\s*"([^"]+)"',
    ]

    for pat in patterns:
        m = re.search(pat, html)
        if m:
            u = m.group(1)
            if is_valid_username(u):
                return u

    return None

def extract_author_url_from_post(driver, post_url: str, max_retries: int = 2) -> Optional[str]:
    """
    主函数：从帖子页面提取作者主页URL
    
    提取顺序：
    1) article header 里找作者链接（最稳）
    2) JSON-LD 解析 author（第二路径）
    3) page_source 正则找 username（第三兜底）
    """
    for attempt in range(max_retries):
        try:
            driver.get(post_url)
            time.sleep(2)  # 等待页面加载

            # 1) 第一路径：从 article header 找作者链接（最准确）
            try:
                elems = driver.find_elements(By.CSS_SELECTOR, "article header a[href]")
                for e in elems:
                    href = e.get_attribute("href") or ""
                    if is_valid_profile_url(href):
                        return href
            except Exception:
                pass

            # 2) 第二路径：JSON-LD解析
            username = extract_author_from_jsonld(driver)
            if username and is_valid_username(username):
                return profile_url_from_username(username)

            # 3) 第三路径：页面源代码正则匹配
            username = extract_author_from_pagesource_regex(driver)
            if username and is_valid_username(username):
                return profile_url_from_username(username)

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
                
    return None

if 'post_urls' not in globals() or not post_urls:
    print("没有找到帖子链接，请先运行获取链接的 cell")
else:
    total = len(post_urls)
    print(f"开始遍历 {total} 个帖子，提取作者主页...\n")

    # 创建存储结果的列表
    results = []
    success_count = 0
    output_filename = f"hashtag_#{keyword}.xlsx"

    for i, post_url in enumerate(post_urls, 1):
        print(f"[{i}/{total}] 访问: {post_url}")
        try:
            author_url = extract_author_url_from_post(driver, post_url)
            
            # 创建结果字典
            result_dict = {
                "URL": post_url,
                "authorURL": author_url if author_url else "未找到"
            }
            
            # 添加到结果列表
            results.append(result_dict)
            
            if author_url:
                success_count += 1
                print(f" 成功: {author_url}")
            else:
                print(" 未找到作者链接")
                
        except Exception as e:
            print(f"  异常: {e}")
            # 即使异常也要记录
            results.append({
                "post_url": post_url,
                "author_url": f"提取失败: {str(e)}"
            })
        
        time.sleep(0.6)  # 避免请求过快

    # 保存到Excel
    if results:
        df = pd.DataFrame(results)
        df.to_excel(output_filename, index=False, engine='openpyxl')
        print(f"\n共处理 {total} 个帖子，成功提取 {success_count} 条作者链接")
        print(f"已保存至: {output_filename}")
        
        # 显示前几行数据
        print("\n数据预览:")
        print(df.head())
    else:
        print("没有数据可保存")

🔍 开始遍历 35 个帖子，提取作者主页...

[1/35] 访问: https://www.instagram.com/p/DAQzdDQP5gY/
  ✅ 成功: https://www.instagram.com/aitommylove/
[2/35] 访问: https://www.instagram.com/p/B9-ASMrl0f8/
  ✅ 成功: https://www.instagram.com/powerpuffgirls/
[3/35] 访问: https://www.instagram.com/p/DCH354MMmTn/
  ✅ 成功: https://www.instagram.com/autumnofthesoul/
[4/35] 访问: https://www.instagram.com/p/DNEZSjzsZMY/
  ✅ 成功: https://www.instagram.com/wasretro/
[5/35] 访问: https://www.instagram.com/p/DGbHoGjxj3p/
  ✅ 成功: https://www.instagram.com/wasretro/
[6/35] 访问: https://www.instagram.com/p/DTg12zgktSa/
  ✅ 成功: https://www.instagram.com/mobilelegendsforlife/
[7/35] 访问: https://www.instagram.com/p/DTbFV7wE0OM/
  ✅ 成功: https://www.instagram.com/tera_princess7/
[8/35] 访问: https://www.instagram.com/p/DNiEUM8M6xi/
  ✅ 成功: https://www.instagram.com/shes_sinister/
[9/35] 访问: https://www.instagram.com/p/DMxt-mRqxFm/
  ✅ 成功: https://www.instagram.com/autumnsfeels/
[10/35] 访问: https://www.instagram.com/p/DNwDs_42COl/
  ✅ 成功: https:/